In [ ]:
## 5折交叉验证

In [7]:
import optuna
import pandas as pd
import numpy as np
import os
import random
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
from xgboost import XGBClassifier
from imblearn.combine import SMOTEENN
import warnings
warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

os.environ['LOKY_MAX_CPU_COUNT'] = '6'

df = pd.read_csv('../数据/训练集.csv')
tezheng = ['ggt', 'drinking', 'ast', 'age', 'hr', 'alt', 'mono', 'gender', 'umalb', 'sua', 'dbil', 'bun', 'tp', 'idbil', 'rbc']

X = df[tezheng]
y = df['ms_cds']
print(f"总数据量: {X.shape}")

def objective(trial):
    params = {
        'objective': 'binary:logistic',
        'eval_metric': 'logloss',
        'verbosity': 0,
        'random_state': SEED,
        'n_jobs': 6,
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.2, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 15),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 100),
        'subsample': trial.suggest_float('subsample', 0.4, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.4, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'gamma': trial.suggest_float('gamma', 1e-8, 1.0, log=True),
    }

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    cv_scores = []

    for train_index, val_index in cv.split(X, y):
        X_train_fold, X_val_fold = X.iloc[train_index], X.iloc[val_index]
        y_train_fold, y_val_fold = y.iloc[train_index], y.iloc[val_index]

        sampler = SMOTEENN(random_state=SEED, n_jobs=-1)
        X_train_res, y_train_res = sampler.fit_resample(X_train_fold, y_train_fold)

        model = XGBClassifier(**params)
        model.fit(X_train_res, y_train_res)

        y_pred = model.predict(X_val_fold)
        score = f1_score(y_val_fold, y_pred)
        cv_scores.append(score)

    return np.mean(cv_scores)


sampler = optuna.samplers.TPESampler(seed=SEED)
study = optuna.create_study(direction='maximize', sampler=sampler)

study.enqueue_trial({
    'n_estimators': 100,
    'learning_rate': 0.1,
    'max_depth': 6,
    'min_child_weight': 1,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'reg_alpha': 1e-8,
    'reg_lambda': 1.0,
    'gamma': 1e-8,
})

print("开始 Optuna 优化 (5折 StratifiedKFold)...")
study.optimize(objective, n_trials=300)

print("-" * 50)
print(f"交叉验证最佳平均 F1-score: {study.best_value:.4f}")
print(f"最佳参数: {study.best_params}")

[I 2026-04-27 21:01:32,848] A new study created in memory with name: no-name-c45f89d0-bf7f-4070-afc7-4f344350d123


总数据量: (4684, 15)
开始 Optuna 优化 (5折 StratifiedKFold)...


[I 2026-04-27 21:01:35,847] Trial 0 finished with value: 0.35203387437338085 and parameters: {'n_estimators': 100, 'learning_rate': 0.1, 'max_depth': 6, 'min_child_weight': 1, 'subsample': 0.8, 'colsample_bytree': 0.8, 'reg_alpha': 1e-08, 'reg_lambda': 1.0, 'gamma': 1e-08}. Best is trial 0 with value: 0.35203387437338085.
[I 2026-04-27 21:01:39,552] Trial 1 finished with value: 0.337399852388938 and parameters: {'n_estimators': 437, 'learning_rate': 0.1667521176194013, 'max_depth': 12, 'min_child_weight': 60, 'subsample': 0.4936111842654619, 'colsample_bytree': 0.49359671220172163, 'reg_alpha': 3.3323645788192616e-08, 'reg_lambda': 0.6245760287469893, 'gamma': 0.0006440507553993703}. Best is trial 0 with value: 0.35203387437338085.
[I 2026-04-27 21:01:44,699] Trial 2 finished with value: 0.2889968993250946 and parameters: {'n_estimators': 737, 'learning_rate': 0.005394455304087533, 'max_depth': 15, 'min_child_weight': 84, 'subsample': 0.5274034664069657, 'colsample_bytree': 0.509094980

--------------------------------------------------
交叉验证最佳平均 F1-score: 0.3832
最佳参数: {'n_estimators': 514, 'learning_rate': 0.11483790087939834, 'max_depth': 14, 'min_child_weight': 1, 'subsample': 0.8684141032164191, 'colsample_bytree': 0.41070399692078213, 'reg_alpha': 0.12015365900335254, 'reg_lambda': 0.00028341147727710273, 'gamma': 3.4567207382526012e-06}


In [14]:
import pandas as pd
import os
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from imblearn.combine import SMOTEENN
import xgboost as xgb

os.environ['LOKY_MAX_CPU_COUNT'] = '6'
tezheng = ['ggt', 'drinking', 'ast', 'age', 'hr', 'alt', 'mono', 'gender', 'umalb', 'sua', 'dbil', 'bun', 'tp', 'idbil', 'rbc']

paras =  {'n_estimators': 514, 'learning_rate': 0.11483790, 'max_depth': 14, 'min_child_weight': 1, 'subsample': 0.86841410, 'colsample_bytree': 0.41070400, 'reg_alpha': 0.12015366, 'reg_lambda': 0.00028341, 'gamma': 3.45672074e-06}
df = pd.read_csv('../数据/训练集.csv')
df1 = pd.read_csv('../数据/测试集.csv')

X_train = df.loc[:, tezheng]
y_train = df.loc[:,'ms_cds']
X_test = df1.loc[:, tezheng]
y_test = df1.loc[:,'ms_cds']

X_train, y_train = SMOTEENN(random_state=42, n_jobs=-1).fit_resample(X_train, y_train)
model = xgb.XGBClassifier(**paras,n_jobs=-1, random_state=42,objective='binary:logistic',eval_metric= 'logloss',verbosity=0)
model.fit(X_train, y_train)

# 预测
y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)[:, 1]

# 计算指标
print(f"准确率: {accuracy_score(y_test, y_pred):.6f}")
print(f"精确率: {precision_score(y_test, y_pred):.6f}")
print(f"召回率: {recall_score(y_test, y_pred):.6f}")
print(f"F1-Score: {f1_score(y_test, y_pred):.6f}")
print(f"AUC值: {roc_auc_score(y_test, y_pred_proba):.6f}")

准确率: 0.849701
精确率: 0.300000
召回率: 0.625000
F1-Score: 0.405405
AUC值: 0.817161


In [ ]:
#不去xgb